In [4]:
import lightly_train
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn.functional as F

# ------------------------
# Dataset
# ------------------------
class ImageMaskDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir  = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
        self.img_paths = sorted([
            p for p in self.img_dir.iterdir() if p.suffix.lower() in IMG_EXTS
        ])
        self.img_paths = [p for p in self.img_paths if (self.mask_dir / p.name).exists()]

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img      = Image.open(img_path).convert("RGB")
        return {
            'image':    transforms.ToTensor()(img),
            'img_name': img_path.name,
        }

# ------------------------
# Prediction
# ------------------------
def predict_batch(model, batch_images):
    if model.training: model.eval()
    device     = next(model.parameters()).device
    pred_masks = []
    img_size   = model.image_size
    resize_dim = [img_size] if isinstance(img_size, int) else [min(img_size)]

    for i in range(batch_images.shape[0]):
        x    = batch_images[i]
        h, w = x.shape[-2:]
        if x.dtype != torch.float32:
            x = x.float()
        x      = transforms.functional.normalize(x, mean=model.image_normalize["mean"],
                                                     std=model.image_normalize["std"])
        x      = transforms.functional.resize(x, size=resize_dim).unsqueeze(0)
        logits = model._forward_logits(x)[:, :-1]
        logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)
        masks  = model.internal_class_to_class[logits.argmax(dim=1)]
        pred_masks.append(masks[0].cpu().numpy())
    return pred_masks

# ------------------------
# Paths & config
# ------------------------
IMG_DIR  = Path("./42folders_7classes_png/val/images")
MASK_DIR = Path("./42folders_7classes_png/val/masks")

CKPT_PATHS = [
    "./out_slurm_vits16/my_experiment_petiole_42tiff_7classes/checkpoints/best.ckpt",
    "./out_slurm_vitb16-eomt/my_experiment_petiole_42tiff_7classes/checkpoints/best.ckpt",
    "./out_slurm_vitl16/my_experiment_petiole_42tiff_7classes/checkpoints/best.ckpt",
    "./out_slurm_vitl16-eomt-cityscapes_new/my_experiment_petiole_42tiff_7classes/checkpoints/best.ckpt",
]

BATCH_SIZE, NUM_WORKERS = 4, 4

def extract_folder_name(ckpt_path):
    for part in Path(ckpt_path).parts:
        if part.startswith("out_slurm_"): return part
    raise ValueError(f"No 'out_slurm_*' found in: {ckpt_path}")

# ------------------------
# Main
# ------------------------
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset    = ImageMaskDataset(IMG_DIR, MASK_DIR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Total images: {len(dataset)} | Batches: {len(dataloader)}")
print("=" * 70)

for ckpt_path in CKPT_PATHS:
    folder_name = extract_folder_name(ckpt_path)
    out_dir     = Path("dino_results") / folder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n>>> {folder_name}")
    model = lightly_train.load_model(ckpt_path)
    model.eval().to(device)

    with torch.inference_mode():
        for batch_idx, batch in enumerate(dataloader):
            pred_masks = predict_batch(model, batch['image'].to(device))

            for i, img_name in enumerate(batch['img_name']):
                Image.fromarray(pred_masks[i].astype(np.uint8)).save(out_dir / img_name)

            print(f"  Batch {batch_idx+1}/{len(dataloader)} done")

    print(f"<<< Done → {out_dir}")
    del model
    torch.cuda.empty_cache()

print("\n✓ All checkpoints processed.")

Total images: 9 | Batches: 3

>>> out_slurm_vits16
  Batch 1/3 done
  Batch 2/3 done
  Batch 3/3 done
<<< Done → dino_results/out_slurm_vits16

>>> out_slurm_vitb16-eomt
  Batch 1/3 done
  Batch 2/3 done
  Batch 3/3 done
<<< Done → dino_results/out_slurm_vitb16-eomt

>>> out_slurm_vitl16
  Batch 1/3 done
  Batch 2/3 done
  Batch 3/3 done
<<< Done → dino_results/out_slurm_vitl16

>>> out_slurm_vitl16-eomt-cityscapes_new
  Batch 1/3 done
  Batch 2/3 done
  Batch 3/3 done
<<< Done → dino_results/out_slurm_vitl16-eomt-cityscapes_new

✓ All checkpoints processed.
